In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely import wkb, wkt
from libpysal.weights import DistanceBand
#import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from libpysal.weights import DistanceBand, lag_spatial
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW


g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
# Replico i filtri: geometrie non vuote e RecordType != "ZIKV"
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()

# ------------------------------------------------------------------
# 3) PREPARA DATASET come in R (selezione variabili e aggregazione)
# ------------------------------------------------------------------
avg_var = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + avg_var +
             ["geometry", "NUTS_NAME", "NUTS_CODE"])

df = g[cols_keep].copy()

# In R raggruppi per geometry, NUTS_NAME, NUTS_CODE; in GeoPandas
# conviene usare l'identificativo NUTS_CODE (che è unico) e conservare la prima geometria.
agg_dict = {"incidence": "sum"}
agg_dict.update({v: "mean" for v in avg_var})
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"})

fin = (
    df.groupby("NUTS_CODE", as_index=False)
      .agg(agg_dict)
      .set_geometry("geometry")
)
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)

# Centroidi per controllo/coords (come in R)
fin["cent"] = fin.geometry.centroid

# ------------------------------------------------------------------
# 4) STANDARDIZZA (z-score) tutte le variabili numeriche
# ------------------------------------------------------------------
num_cols = fin.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
fin_scaled = fin.copy()
fin_scaled[num_cols] = scaler.fit_transform(fin_scaled[num_cols])

# ------------------------------------------------------------------
# 5) RIPROIEZIONE a metrica (EPSG:32633 ~ UTM 33N), come in R
# ------------------------------------------------------------------
fin_m = fin_scaled.to_crs(32633)

# ------------------------------------------------------------------
# 6) COSTRUISCI matrici di vicinato entro 150 km e Y laggata
# ------------------------------------------------------------------
# In R, coordinates() sullo Spatial di poligoni restituisce i centroidi.
# Qui uso i centroidi in metri (proiezione UTM 33N).
coords = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

# threshold 150 km, binaria, senza diagonale
W = DistanceBand(coords, threshold=150000, binary=True, silence_warnings=True)
# trasformo in matrice sparsa CSR (se serve), o applico direttamente come W * y
# libpysal conserva W in forma weights + neighbors; uso full() per matrice densa se serve
W_sparse = W.sparse  # scipy CSR matrix

# y laggata
y = fin_m["incidence"].to_numpy().reshape(-1, 1)
y_lag = W_sparse.dot(y).ravel()
fin_m["Y_weighted"] = y_lag

# ------------------------------------------------------------------
# 7) STIMA OLS con formula analoga all’R (Mean_T_Summer esclusa come nel codice R)
# ------------------------------------------------------------------
# NOTA: qui stai regredendo Y_lag su X (non è una SAR/SLX); è OLS classico.
formula = ("Y_weighted ~ Forest + Shrub + Urban + Crop + Water + Other + "
           "Mean_P_Winter + Max_WD_Winter + Max_DD_Winter + "
           "Mean_T_Spring + Max_DD_Spring + "
           # "Mean_T_Summer + "  # esclusa come in R
           "Max_WD_Autumn + Max_DD_Autumn + gdp + pop_dens")

# statsmodels richiede un DataFrame "piatto" (niente geometry)
Xy = pd.DataFrame(fin_m.drop(columns=["geometry", "cent"]))


# ------------------------------------------------------------------
# 8) VIF per multicollinearità globale (come car::vif)
# ------------------------------------------------------------------
# Creo la matrice di design senza l'intercetta per VIF
design_cols = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
               "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
               "Mean_T_Spring", "Max_DD_Spring",
               "Max_WD_Autumn", "Max_DD_Autumn",
               "gdp", "pop_dens"]




kernels = ["gaussian", "bisquare", "exponential"]   # add/remove kernels as you like

coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

def get_aicc_safe(fit):
    # Try common names across mgwr versions
    for name in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, name):
            try:
                return float(getattr(fit, name))
            except Exception:
                pass
    # Fallback to AIC if nothing else exists
    if hasattr(fit, "aic"):
        try:
            return float(getattr(fit, "aic"))
        except Exception:
            pass
    return np.nan

rows = []
best = None
thresholds = list(range(100_000, 200_001, 10_000))   # 100–200 km by 10 km
eps = 1e-9
for d in thresholds:
    # 1) Build Y_weighted EXACTLY like your OLS at this distance
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)
    y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1)
    y_lag = Wd.sparse.dot(y_raw).ravel()

    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std
    
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 2) Complete-case mask and align arrays
    need_cols = ["Y_weighted"] + design_cols
    mask = Xy_thr[need_cols].notna().all(axis=1).to_numpy()
    if mask.sum() == 0:
        print(f"[WARN] {d/1000:.0f} km: no complete cases. Skipping.")
        continue

    coords = coords_all[mask]
    y = Xy_thr.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)
    X = Xy_thr.loc[mask, design_cols].to_numpy()
    n = X.shape[0]

    row_sums = np.asarray(Wd.sparse.sum(axis=1)).ravel()
    isolated = int((row_sums[mask] == 0).sum())

    for ker in kernels:
        try:
            # Bandwidth search (adaptive neighbor count)
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search()

            fit = GWR(coords, y, X, bw,
                      kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            mean_r2 = float(np.mean(fit.localR2)) if hasattr(fit, "localR2") else np.nan
            min_r2  = float(np.min(fit.localR2))  if hasattr(fit, "localR2") else np.nan
            max_r2  = float(np.max(fit.localR2))  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,
                "bandwidth_neighbors": int(bw) if bw is not None else np.nan,
                "AICc": aicc_val,
                "mean_localR2": mean_r2,
                "min_localR2":  min_r2,
                "max_localR2":  max_r2,
                "n": n,
                "isolated_for_Y": isolated
            }
            rows.append(row)

            # Track best by AICc if finite
            if np.isfinite(row["AICc"]) and (best is None or row["AICc"] < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={bw} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={mean_r2:.3f}/{min_r2:.3f}/{max_r2:.3f} | "
                  f"n={n} isoY={isolated}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": n, "isolated_for_Y": isolated, "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")

# Save summary
res_df = pd.DataFrame(rows)

Q:\UserTemp\massaro\AppData\Local\Temp\272\ipykernel_64664\3027866553.py:50: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fin["cent"] = fin.geometry.centroid



=== Coefficient table ===
    Covariate  Estimate   SE p-value
    Intercept     -4.12 0.25   <.001
       Forest      0.19 0.34   0.581
        Shrub      1.52 0.29   <.001
        Urban      0.76 0.27   0.004
         Crop      0.57 0.33   0.091
        Water      0.76 0.27   0.005
        Other      0.54 0.30   0.069
Mean_P_Winter      3.94 0.39   <.001
Max_WD_Winter     -2.42 0.52   <.001
Max_DD_Winter      0.09 0.41   0.819
Mean_T_Spring      2.94 0.45   <.001
Max_DD_Spring     -2.98 0.37   <.001
Max_WD_Autumn     -1.43 0.43   <.001
Max_DD_Autumn      0.94 0.45   0.036
          gdp     -2.48 0.32   <.001
     pop_dens      1.04 0.27   <.001
d=100 km | gaussian   | bw=73.0 | AICc=2041.3346344392476 | R2 μ/lo/hi=0.742/0.442/0.867 | n=1134 isoY=27
d=100 km | bisquare   | bw=100.0 | AICc=1888.9123553840982 | R2 μ/lo/hi=0.764/-1.713/0.963 | n=1134 isoY=27
d=100 km | exponential | bw=73.0 | AICc=2059.631755189019 | R2 μ/lo/hi=0.751/0.492/0.858 | n=1134 isoY=27
d=110 km | gaussian   | 

In [3]:
# ===============================
# GWR with z-scored X and z-scored Wy (after lag)
# ===============================
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from libpysal.weights import DistanceBand
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR

# -----------------
# Read & prepare
# -----------------
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
g  = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()

design_cols = [
    "Forest","Shrub","Urban","Crop","Water","Other",
    "Mean_P_Winter","Max_WD_Winter","Max_DD_Winter",
    "Mean_T_Spring","Max_DD_Spring",
    # "Mean_T_Summer",  # include if you want it
    "Max_WD_Autumn","Max_DD_Autumn",
    "gdp","pop_dens"
]
response_col = "incidence"

cols_keep = ["geometry","NUTS_NAME","NUTS_CODE", response_col] + design_cols
df = g[cols_keep].copy()

# Aggregate: incidence=sum; predictors=mean
agg = {response_col: "sum", **{c: "mean" for c in design_cols}, "geometry": "first", "NUTS_NAME": "first"}
fin = (
    df.groupby("NUTS_CODE", as_index=False)
      .agg(agg)
      .set_geometry("geometry")
)
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)

# Project to metric CRS and build coords (NO SCALING here)
fin_m = fin.to_crs(32633)
cent = fin_m.geometry.centroid
coords_all = np.c_[cent.x, cent.y].astype(float)

# (Optional) tiny jitter if you have exact duplicate centroids
if np.unique(coords_all, axis=0).shape[0] < coords_all.shape[0]:
    rng = np.random.default_rng(42)
    coords_all = coords_all + rng.normal(0.0, 1e-6, size=coords_all.shape)

# -----------------
# Helpers
# -----------------
def get_aicc_safe(fit):
    for nm in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, nm):
            try:
                return float(getattr(fit, nm))
            except Exception:
                pass
    return float(getattr(fit, "aic", np.nan)) if hasattr(fit, "aic") else np.nan

# -----------------
# Loop (distance × kernel)
# -----------------
kernels    = ["gaussian", "bisquare", "exponential"]
thresholds = list(range(100_000, 200_001, 10_000))  # meters (100–200 km)

rows = []
best = None

for d in thresholds:
    # 1) Build binary DistanceBand at distance d (meters) – SUM lag (no row-standardization)
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)

    # 2) Spatial lag of raw incidence (SUM of neighbors), then z-score Wy
    y_raw = fin_m[response_col].to_numpy().reshape(-1, 1).astype(float)
    y_lag = Wd.sparse.dot(y_raw).ravel()

    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {d/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std

    # 3) Assemble modeling frame (raw X, z-scored Wy), drop incomplete rows
    tmp = fin_m[design_cols].copy()
    tmp["Y_weighted"] = y_lag_z
    need_cols = ["Y_weighted"] + design_cols
    Xy_thr = tmp.dropna(subset=need_cols).copy()

    # 4) Align coords/X/y with mask
    mask = Xy_thr.index.to_numpy()
    coords = coords_all[mask]
    y = Xy_thr["Y_weighted"].to_numpy().reshape(-1, 1)

    # 5) Standardize X **now**, on the rows used in the fit
    scaler = StandardScaler().fit(Xy_thr[design_cols])
    X = scaler.transform(Xy_thr[design_cols])
    n, k = X.shape

    # 6) Adaptive bandwidth with a safe minimum to avoid singular locals
    bw_min = max(k + 1 + 5, 30)      # #params (k + intercept) + cushion; at least ~30 neighbors
    bw_min = min(bw_min, n - 1)

    for ker in kernels:
        try:
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search(bw_min=bw_min)
            if isinstance(bw, (int, float)) and bw < bw_min:
                bw = bw_min

            fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            r2_mu = float(fit.localR2.mean()) if hasattr(fit, "localR2") else np.nan
            r2_lo = float(fit.localR2.min())  if hasattr(fit, "localR2") else np.nan
            r2_hi = float(fit.localR2.max())  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,
                "bandwidth_neighbors": int(bw) if bw is not None else np.nan,
                "AICc": aicc_val,
                "mean_localR2": r2_mu, "min_localR2": r2_lo, "max_localR2": r2_hi,
                "n": int(n),
                "isolated_for_Y": int((np.asarray(Wd.sparse.sum(axis=1)).ravel()[mask] == 0).sum())
            }
            rows.append(row)

            # Track best by AICc
            if np.isfinite(aicc_val) and (best is None or aicc_val < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={row['bandwidth_neighbors']} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={r2_mu:.3f}/{r2_lo:.3f}/{r2_hi:.3f} | "
                  f"n={n} isoY={row['isolated_for_Y']}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": int(n), "isolated_for_Y": None, "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")

# Summary dataframe
res_df = pd.DataFrame(rows)


d=100 km | gaussian   | bw=32 | AICc=2204.2230279990354 | R2 μ/lo/hi=-42.389/-7505.783/0.867 | n=1134 isoY=27


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=100 km | bisquare   | bw=100 | AICc=2396.3925363168737 | R2 μ/lo/hi=-inf/-inf/0.932 | n=1134 isoY=27
d=100 km | exponential | bw=31 | AICc=2188.4354745155797 | R2 μ/lo/hi=0.660/-1.501/0.856 | n=1134 isoY=27
d=110 km | gaussian   | bw=32 | AICc=2145.183026419166 | R2 μ/lo/hi=-42.809/-7597.269/0.886 | n=1134 isoY=11


KeyboardInterrupt: 

In [5]:
# ... your prep above ...

for d in thresholds:
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)

    y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1).astype(float)
    y_lag = Wd.sparse.dot(y_raw).ravel()

    # z-score Wy (global)
    y_lag_z = (y_lag - y_lag.mean()) / y_lag.std(ddof=0)

    # build X/y/coords (align by complete cases)
    tmp = fin_m[design_cols].copy()
    tmp["Y_weighted"] = y_lag_z
    need_cols = ["Y_weighted"] + design_cols
    Xy_thr = tmp.dropna(subset=need_cols).copy()

    mask = Xy_thr.index.to_numpy()
    coords = coords_all[mask]
    y = Xy_thr["Y_weighted"].to_numpy().reshape(-1, 1)

    # EITHER keep raw X (matches your original behavior)...
    X = Xy_thr[design_cols].to_numpy()

    # ...OR standardize X on the used rows (if you prefer SD units)
    # from sklearn.preprocessing import StandardScaler
    # scaler = StandardScaler().fit(Xy_thr[design_cols])
    # X = scaler.transform(Xy_thr[design_cols])

    n, k = X.shape

    # ******* KEY CHANGE *******
    # force a safe minimum bandwidth ~ your earlier 73 neighbors
    bw_min = max(70, k + 1 + 10)         # ≥70 or ≥(#params+10), whichever is larger
    bw_min = min(bw_min, n - 1)

    for ker in kernels:
        try:
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search(bw_min=bw_min)
            # enforce bw_min if selector returns smaller
            if isinstance(bw, (int, float)) and bw < bw_min:
                bw = bw_min

            fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = getattr(fit, "aicc", getattr(fit, "AICc", np.nan))
            r2_mu = float(fit.localR2.mean()); r2_lo = float(fit.localR2.min()); r2_hi = float(fit.localR2.max())
            print(f"d={d/1000:.0f} km | {ker:10s} | bw={int(bw)} | "
                  f"AICc={aicc_val:.1f} | R2 μ/lo/hi={r2_mu:.3f}/{r2_lo:.3f}/{r2_hi:.3f} | n={n}")
        except Exception as e:
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")


d=100 km | gaussian   | bw=71 | AICc=2465.7 | R2 μ/lo/hi=-1.059/-49.230/0.777 | n=1134


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=100 km | bisquare   | bw=100 | AICc=2396.4 | R2 μ/lo/hi=-inf/-inf/0.932 | n=1134
d=100 km | exponential | bw=71 | AICc=2459.0 | R2 μ/lo/hi=0.605/0.358/0.763 | n=1134
d=110 km | gaussian   | bw=71 | AICc=2389.3 | R2 μ/lo/hi=-1.207/-54.856/0.800 | n=1134


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=110 km | bisquare   | bw=100 | AICc=2308.9 | R2 μ/lo/hi=-inf/-inf/0.947 | n=1134
d=110 km | exponential | bw=71 | AICc=2394.9 | R2 μ/lo/hi=0.624/0.356/0.787 | n=1134
d=120 km | gaussian   | bw=71 | AICc=2348.0 | R2 μ/lo/hi=-1.149/-55.158/0.805 | n=1134


...

d=180 km | bisquare   | bw=86 | AICc=2025.2 | R2 μ/lo/hi=-inf/-inf/0.980 | n=1134
d=180 km | exponential | bw=71 | AICc=2133.8 | R2 μ/lo/hi=0.676/0.370/0.833 | n=1134
d=190 km | gaussian   | bw=71 | AICc=2182.0 | R2 μ/lo/hi=-0.396/-24.806/0.851 | n=1134


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=190 km | bisquare   | bw=86 | AICc=1987.7 | R2 μ/lo/hi=-inf/-inf/0.980 | n=1134
d=190 km | exponential | bw=71 | AICc=2115.2 | R2 μ/lo/hi=0.679/0.375/0.835 | n=1134
d=200 km | gaussian   | bw=71 | AICc=2141.5 | R2 μ/lo/hi=-0.295/-22.655/0.852 | n=1134


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=200 km | bisquare   | bw=108 | AICc=2011.6 | R2 μ/lo/hi=-inf/-inf/0.964 | n=1134
d=200 km | exponential | bw=71 | AICc=2069.0 | R2 μ/lo/hi=0.686/0.381/0.836 | n=1134
